# llms.txt generation

> Build `llms.txt` from `pkg/skill.py`

In [ ]:
#| default_exp llms

In [ ]:
from fastcore.utils import *
from fastcore.script import call_parse
from pyskills.core import _first_para, _mod_docstr, norm_docs
from fastcore.pyparse import parse_py_file

In [ ]:
from fastcore.test import test_eq
from tempfile import TemporaryDirectory

A package's `skill.py` is the single source of truth for LLM documentation: `doc(pkg.skill)` renders it in Python, and this module renders the same file as `llms.txt` for web readers without importing it. Skill docstrings use `###`+ headings so `# pkg` and `## Section` stay reserved for `llms.txt`.

## Parsing `skill.py`

`parse_skill` reads a `skill.py` file via AST, splitting its docstring into a one-line `summary` and `body`, and reading literal `__docs__` and `__base_url__` assignments.

In [ ]:
def parse_skill(
    path:str|Path # Path to a `skill.py` file
)->AttrDict: # `AttrDict(summary, body, docs, base_url)`
    "Parse `skill.py`'s docstring and literal `__docs__` / `__base_url__` via AST without importing it"
    m = parse_py_file(path)
    paras = m.doc.split('\n\n') if m.doc else []
    return AttrDict(summary=' '.join(paras[0].split()) if paras else '',
        body='\n\n'.join(paras[1:]).strip() if len(paras)>1 else '',
        docs=m.literals.get('__docs__'), base_url=m.literals.get('__base_url__', ''))

In [ ]:
sk = parse_skill(Path('../pyskills/skill.py'))
assert sk.summary and sk.body and 'Docs' in sk.docs
sk.summary, sk.docs

## Formatting reference sections

`_mod_line` formats one module cross-reference using its AST docstring summary and `mod_urls` entry. Pure-docstring modules without a web page, such as `guidance.py` or `createskill.py`, emit a sibling `llms-<name>.txt` companion file copied verbatim by Quarto.

In [ ]:
_MARKER = '<!-- Generated by pyskills-llms; edit {pkg}/skill.py, not this file -->'

def _mod_doc(mod_nm, pkg_dir, pkg_name):
    if mod_nm.startswith(pkg_name + '.'):
        base = pkg_dir.joinpath(*mod_nm[len(pkg_name)+1:].split('.'))
        for p in (base.with_suffix('.py'), base/'__init__.py'):
            if p.exists(): return m.doc if (m := parse_py_file(p, safe=True)) else ''
    return _mod_docstr(mod_nm)

def _mod_line(nm, pkg_dir, pkg_name, urls, comps):
    doc = _mod_doc(nm, pkg_dir, pkg_name)
    summ = _first_para(doc)
    href = urls.get(nm, '')
    if not href and doc:
        href = f"llms-{nm.split('.')[-1]}.txt"
        comps[href] = f'{doc.strip()}\n\n{_MARKER.format(pkg=pkg_name)}\n'
    lbl = f'[{nm}]({href})' if href else f'`{nm}`'
    return f'- {lbl}: {summ}' if summ else f'- {lbl}' 

In [ ]:
comps = {}
ln = _mod_line('pyskills.createskill', Path('../pyskills').resolve(), 'pyskills', {}, comps)
assert '[pyskills.createskill](llms-createskill.txt):' in ln and list(comps) == ['llms-createskill.txt']
ln

`_fmt_sect` renders one `## Section` block, mixing module entries and `(title, href, desc)` link tuples.

In [ ]:
def _fmt_sect(sec, its, pkg_dir, pkg_name, urls, comps):
    lines = [_mod_line(it[1], pkg_dir, pkg_name, urls, comps) if it[0]=='mod'
        else (f'- [{it[1]}]({it[2]}): {it[3]}' if it[3] else f'- [{it[1]}]({it[2]})') for it in its]
    return f'## {sec}\n\n' + '\n'.join(lines)

In [ ]:
_fmt_sect('Docs', [('mod', 'pyskills.core'), ('link', 'Guide', 'guide.html.md', 'Intro')],
    Path('../pyskills').resolve(), 'pyskills', {'pyskills.core': 'core.html.md'}, {})

## Building `llms.txt`

`pkg_llms` assembles the title, summary blockquote, normalized `Base URL:`, skill body, reference sections, optional `## API` link (passed by `fastship` `ship-docs`), and trailing marker.

In [ ]:
def pkg_llms(
    path:str|Path, # Package directory containing `skill.py`
    base_url:str='', # Published docs base URL (normalized to end with `/`)
    mod_urls:dict=None, # Optional `{module_name: rel_url}` for modules with web doc pages
    api_url:str=None, # Optional relative URL for a top-level API page (e.g. `api.html.md`)
)->tuple[str, dict[str, str]]: # `(llms_txt, {companion_filename: content})`
    "Build `llms.txt` and pure-docstring companion files from `path/skill.py` without importing it"
    pkg_dir = Path(path).expanduser().resolve()
    sk_path = pkg_dir/'skill.py'
    if not sk_path.exists(): return '', {}
    pkg_name,sk,urls,comps = pkg_dir.name,parse_skill(sk_path),mod_urls or {},{}
    base = (sk.base_url or base_url).strip()
    if base: base = base.rstrip('/') + '/'
    parts = [f'# {pkg_name}']
    if sk.summary: parts.append(f'> {sk.summary}')
    if base: parts.append(f'Base URL: {base}')
    if sk.body: parts.append(sk.body)
    sects = norm_docs(sk.docs, pkg_dir, pkg_name, mod_urls=urls or None)
    opt = next((its for s,its in sects if s=='Optional'), None)
    for sec,its in sects:
        if sec != 'Optional': parts.append(_fmt_sect(sec, its, pkg_dir, pkg_name, urls, comps))
    if api_url: parts.append(f'## API\n\n- [API reference]({api_url}): Public API for `{pkg_name}`')
    if opt: parts.append(_fmt_sect('Optional', opt, pkg_dir, pkg_name, urls, comps))
    parts.append(_MARKER.format(pkg=pkg_name))
    return '\n\n'.join(parts) + '\n', comps

In [ ]:
txt,comps = pkg_llms('../pyskills', base_url='https://AnswerDotAI.github.io/pyskills',
    mod_urls={'pyskills.core': 'core.html.md', 'pyskills.llms': 'llms.html.md'})
assert 'Base URL: https://AnswerDotAI.github.io/pyskills/' in txt
assert '- [pyskills.core](core.html.md):' in txt and list(comps) == ['llms-createskill.txt']
txt[:500]

## Writing and updating `llms.txt`

`update_llms_txt` writes `llms.txt` and companion `llms-*.txt` files when absent or generated, leaves hand-written files untouched, and removes stale generated `llms-*.txt` files when `skill.py` or a companion disappears.

In [ ]:
def _is_gen(p:Path): return p.exists() and '<!-- Generated by ' in p.read_text(encoding='utf-8')

def _write_changed(p:Path, txt:str):
    if p.exists() and p.read_text(encoding='utf-8')==txt: return False
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt, encoding='utf-8')
    return True

def _rm_gen(p:Path):
    if not _is_gen(p): return False
    p.unlink()
    return True

In [ ]:
def update_llms_txt(
    path:str|Path, # Package directory containing `skill.py`
    fn:str|Path=None, # Output `llms.txt` path (defaults to `<path>/../llms.txt`)
    base_url:str='', # Published docs base URL
    mod_urls:dict=None, # Optional `{module_name: rel_url}` map
    api_url:str=None, # Optional relative URL for a top-level API page
)->bool: # True if any file was written or removed
    "Write `llms.txt` and companion files when absent or generated; remove stale generated files"
    pkg_dir = Path(path).expanduser().resolve()
    fn = Path(fn).expanduser() if fn else pkg_dir.parent/'llms.txt'
    if fn.exists() and not _is_gen(fn): return False
    txt,comps = pkg_llms(pkg_dir, base_url=base_url, mod_urls=mod_urls, api_url=api_url)
    changed = _rm_gen(fn) if not txt else _write_changed(fn, txt)
    for nm,body in comps.items():
        p = fn.parent/nm
        if not p.exists() or _is_gen(p): changed |= _write_changed(p, body)
    if fn.parent.exists():
        for p in fn.parent.glob('llms-*.txt'):
            if p.name not in comps: changed |= _rm_gen(p)
    return changed

Create a small temporary package with `skill.py`, `guidance.py`, `core.py`, a re-export barrel, a private module, and a stale generated `llms-api.txt`.

In [ ]:
_td = TemporaryDirectory()
d = Path(_td.name)
pkg = d/'demo'
pkg.mkdir()
(pkg/'skill.py').write_text('"""Demo summary.\n\n### Usage\n\n- fast\n"""\n__docs__ = {"Optional": [("Extra", "extra.html.md", "More")]}\n')
(pkg/'guidance.py').write_text('"""Domain rules.\n\nBe careful.\n"""\n')
(pkg/'core.py').write_text('"""Core helpers."""\ndef f(): pass\n')
(pkg/'barrel.py').write_text('from .core import *\n')
(pkg/'_priv.py').write_text('"""Hidden."""\ndef g(): pass\n')
(d/'llms-api.txt').write_text('old\n<!-- Generated by pyskills-llms; edit demo/skill.py, not this file -->\n')

Running `update_llms_txt` writes `llms.txt` and `llms-guidance.txt`, skips `barrel.py` and `_priv.py`, and deletes the stale `llms-api.txt`. A second identical call returns `False`.

In [ ]:
fn = d/'llms.txt'
kw = dict(base_url='https://example.com/demo', mod_urls={'demo.core': 'core.html.md'}, api_url='api.html.md')
assert update_llms_txt(pkg, fn, **kw) and not update_llms_txt(pkg, fn, **kw)
out = fn.read_text()
assert '- [demo.guidance](llms-guidance.txt): Domain rules.' in out and '- [API reference](api.html.md):' in out
assert '- [demo.core](core.html.md): Core helpers.' in out and 'barrel' not in out and '_priv' not in out
assert (d/'llms-guidance.txt').exists() and not (d/'llms-api.txt').exists()
out

Removing `skill.py` cleans up generated files, while a hand-written `llms.txt` without a generator marker is never overwritten.

In [ ]:
(pkg/'skill.py').unlink()
assert update_llms_txt(pkg, fn) and not fn.exists() and not (d/'llms-guidance.txt').exists()
fn.write_text('# Hand written\n')
assert not update_llms_txt(pkg, fn) and fn.read_text() == '# Hand written\n'
_td.cleanup()

## Command line

`pyskills-llms` exposes `update_llms_txt` as a CLI.

In [ ]:
@call_parse
def llms_cli(
    path:str, # Package directory containing `skill.py`
    out:str=None, # Output `llms.txt` path (defaults to `<path>/../llms.txt`)
    base_url:str='', # Published docs base URL
    api_url:str=None, # Optional relative URL for a top-level API page
):
    "Generate `llms.txt` and companion files from `path/skill.py`"
    update_llms_txt(path, fn=out, base_url=base_url, api_url=api_url)